<a href="https://colab.research.google.com/github/andreagrioni/Tutorials/blob/master/Filter_Bed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Filter BED notebook

short script to extend boundaries of a GTF file to arbitrary size and intersect it with user BED files stored into GitHub repo.

## Mount Google Drive

No more required.

In [0]:
# from google.colab import drive
# drive.mount('/content/drive')

## install dependency, download data and annotation

In [6]:
! git clone https://github.com/andreagrioni/Tutorials.git
! apt-get install bedtools
! mkdir results
! mkdir annotation
! wget ftp://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/gencode.v32.primary_assembly.annotation.gtf.gz -P annotation/

Cloning into 'Tutorials'...
remote: Enumerating objects: 132, done.
remote: Total 132 (delta 0), reused 0 (delta 0), pack-reused 132
Receiving objects: 100% (132/132), 437.65 KiB | 8.42 MiB/s, done.
Resolving deltas: 100% (66/66), done.
Reading package lists... Done
Building dependency tree       
Reading state information... Done
bedtools is already the newest version (2.26.0+dfsg-5).
The following package was automatically installed and is no longer required:
  libnvidia-common-430
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.
mkdir: cannot create directory ‘results’: File exists
mkdir: cannot create directory ‘annotation’: File exists
--2020-01-13 14:59:26--  ftp://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/gencode.v32.primary_assembly.annotation.gtf.gz
           => ‘annotation/gencode.v32.primary_assembly.annotation.gtf.gz.1’
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.197.74
Connecting to ftp.ebi.ac.uk (f

## Filter annotation for UTR

In [0]:
! zcat annotation/gencode.v32.primary_assembly.annotation.gtf.gz | awk '$3=="UTR" {print $0}' > annotation/gencode.v32.primary_assembly.annotation.UTR.gtf

## Arbitrary Extend UTR coordinates

In [0]:
def resize_intervals(annotation, extend, strand=False):
  """ 
  extend annotaton start-end boundaries of
  arbitrary size.

  paramenters:
  annotation=path to annotation file
  extend=nt size to extend
  strand=extend the coding strand (False)
  """
  if strand == False:
    output_name = f'utr_extended_{extend}_unstranded.gtf'
  elif strand == True:
    output_name = f'utr_extended_{extend}_stranded.gtf'

  with open(annotation, 'r') as f_input, open(output_name, "w") as f_output:
    for index, line in enumerate(f_input):
      columns = line.split('\t')
      if strand == False:
        columns[3] = str(int(columns[3]) - extend)
        columns[4] = str(int(columns[4]) + extend)
      elif strand==True:
        if columns[6] == "+":
          columns[4] = str(int(columns[4]) + extend)
        elif columns[6] == "-":
          columns[3] = str(int(columns[3]) - extend)
      new_line = '\t'.join(columns)
      f_output.write(new_line)

  return output_name



## Intersect with BedTools

In [0]:
import subprocess


def run_bedtools(candidate_file, annotation, outdir, strand):
  if strand == True:
    candidate_file_name = os.path.basename(candidate_file).replace('.bed', '.filtered.stranded.bed')
  elif strand == False:
    candidate_file_name = os.path.basename(candidate_file).replace('.bed', '.filtered.unstranded.bed')

  outname = f'{outdir}/{candidate_file_name}'
  cmd = f'bedtools intersect -a {candidate_file} -b {annotation} -u > {outname}'
  try:
    subprocess.run(cmd, shell=True)
  except:
    return "error"

## run code

In [53]:
import os


annotation_path = "annotation/gencode.v32.primary_assembly.annotation.UTR.gtf"
bed_dir_path = "Tutorials/data/filter_bed/"
output_dir_path = "results/"



# extend_size paramenter controls how much extend the UTR
extend_size = 1000
# strand paramenter control if to extend both side of UTR (strand=False) or
# to extend only in the coding direction (strand=True)
strand=False

# extend UTR boundaries
resized_file = resize_intervals(annotation_path, extend_size, strand)
# filter candidates
for bed_file in os.listdir(bed_dir_path):
  if not bed_file.endswith('.bed'):
    continue
  candidates_bed_path = os.path.join(bed_dir_path, bed_file)
  print(candidates_bed_path)
  run_bedtools(candidates_bed_path, resized_file, output_dir_path, strand)


Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.25.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.7.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.8.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.75.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.6.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.45.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.85.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.2.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.3.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.1.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.4.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.9.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.55.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.95.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.5.bed
Tutorials/data/filter_bed/hotspots

## Download Results

This section allows the user to download results

In [57]:
from google.colab import files
from zipfile import ZipFile


with ZipFile('./results_extend_filtered.zip', 'w') as zipObj:
  for bed_file in os.listdir(output_dir_path):
    bed_path = os.path.join(output_dir_path, bed_file)
    print(bed_path)
    zipObj.write(bed_path)

files.download('./results_extend_filtered.zip')

results/hotspots.unstranded.threshold.0.3.filtered.unstranded.bed
results/hotspots.unstranded.threshold.0.85.filtered.stranded.bed
results/hotspots.unstranded.threshold.0.35.filtered.stranded.bed
results/hotspots.unstranded.threshold.0.4.filtered.stranded.bed
results/hotspots.unstranded.threshold.0.9.filtered.unstranded.bed
results/hotspots.unstranded.threshold.0.2.filtered.unstranded.bed
results/hotspots.unstranded.threshold.0.25.filtered.unstranded.bed
results/hotspots.unstranded.threshold.0.85.filtered.unstranded.bed
results/hotspots.unstranded.threshold.0.8.filtered.stranded.bed
results/hotspots.unstranded.threshold.0.6.filtered.stranded.bed
results/hotspots.unstranded.threshold.0.1.filtered.stranded.bed
results/hotspots.unstranded.threshold.0.35.filtered.unstranded.bed
results/hotspots.unstranded.threshold.0.65.filtered.unstranded.bed
results/hotspots.unstranded.threshold.0.65.filtered.stranded.bed
results/hotspots.unstranded.threshold.0.75.filtered.unstranded.bed
results/hotspots